In [7]:
import requests
import pandas as pd
import re
from typing import List, Dict

In [8]:
# Simplest query: no history, just return assistant text


BASE_URL = "http://localhost:8000"

def query_chat(message: str) -> str:
    resp = requests.post(f"{BASE_URL}/chat", json={"message": message}, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    # Prefer last assistant from conversation_history if provided
    ch = data.get("conversation_history")
    if isinstance(ch, list):
        for m in reversed(ch):
            if m.get("role") == "assistant" and str(m.get("content", "")).strip():
                return m["content"]
    # Fallback to single message field
    return str(data.get("message", ""))

In [16]:
# Fix: robust computation accuracy (no division by zero, safe empty expected list)
import re
from typing import List

def _extract_numbers(text: str) -> List[float]:
    return [float(x.replace(',', '.')) for x in re.findall(r"-?\d+(?:[.,]\d+)?", text or "")]


def computation_accuracy_check(
    response: str,
    expected_values: List[float],
    rel_tolerance: float = 0.05,
    abs_tolerance: float = 1e-6,
) -> float:
    """
    Returns fraction of expected values found in response within tolerance.
    - If expected == 0, use absolute tolerance (abs(n) <= abs_tolerance)
    - Else use relative tolerance: abs(n - expected) / max(abs(expected), 1e-12) <= rel_tolerance
    - If expected_values is empty, returns 0.0
    """
    nums_in_response = _extract_numbers(response)
    if not expected_values:
        return 0.0

    correct = 0
    for expected in expected_values:
        if abs(expected) <= abs_tolerance:
            # zero (or near-zero) expected: absolute comparison
            match = any(abs(n) <= abs_tolerance for n in nums_in_response)
        else:
            # non-zero expected: relative comparison
            denom = max(abs(expected), 1e-12)
            match = any(abs(n - expected) / denom <= rel_tolerance for n in nums_in_response)
        if match:
            correct += 1

    return correct / len(expected_values)

print("computation_accuracy_check redefined ✔")



computation_accuracy_check redefined ✔


In [22]:
# Example 
print(query_chat("O seller 3442f8959a84dea7ee197c632cb2df15 é confiável?"))

Não confiável.
- Atrasos: 33,33% (limite aceitável: <= 5%)
- Nota média: 3,0/5,0 (abaixo de 3,5)
- Amostra: 3 pedidos (1 atrasado)


In [9]:
# ============================================================
# 2️⃣ Função de avaliação (LLM como juiz)
# ============================================================

def llm_judge(question: str, answer: str) -> float:
    """
    Usa o próprio modelo (via API local) para avaliar clareza e relevância da resposta.
    Retorna uma nota entre 0 e 1.
    """
    judge_prompt = f"""
    Avalie a seguinte resposta com base em CLAREZA e RELEVÂNCIA (nota de 0 a 1).

    Critérios:
    - 1.0 = resposta completa, clara e diretamente relacionada à pergunta.
    - 0.5 = resposta parcialmente correta ou incompleta.
    - 0.0 = resposta incorreta, confusa ou fora do contexto.

    Pergunta: {question}
    Resposta: {answer}

    Retorne apenas a nota numérica (ex: 0.8)
    """
    try:
        resp = requests.post(f"{BASE_URL}/chat", json={"message": judge_prompt}, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        content = data.get("message") or str(data)
        # extrai primeiro número decimal encontrado
        import re
        match = re.search(r"([01](?:\.\d+)?)", str(content))
        return float(match.group(1)) if match else 0.0
    except Exception:
        return 0.0


In [13]:
# ============================================================
# 3️⃣ Métricas automáticas (fidelidade e precisão)
# ============================================================

def data_consistency_check(response: str, reference_terms: List[str]) -> float:
    """Mede se termos esperados (entidades, palavras-chave) aparecem na resposta."""
    response_lower = response.lower()
    matches = sum(1 for t in reference_terms if t.lower() in response_lower)
    return matches / len(reference_terms) if reference_terms else 1.0


def computation_accuracy_check(response: str, expected_values: List[float], tolerance: float = 0.02) -> float:
    """Compara valores numéricos esperados com os extraídos da resposta."""
    nums_in_response = [float(n) for n in re.findall(r'\d+(?:\.\d+)?', response)]
    if not expected_values:
        return 1.0
    correct = 0
    for expected in expected_values:
        if any(abs(n - expected) / expected < tolerance for n in nums_in_response):
            correct += 1
    return correct / len(expected_values)


def evaluate_response(response: str, reference_terms: List[str], expected_values: List[float]) -> Dict[str, float]:
    data_score = data_consistency_check(response, reference_terms)
    comp_score = computation_accuracy_check(response, expected_values)
    overall = (data_score + comp_score) / 2
    return {
        "Data_Consistency": round(data_score, 3),
        "Computation_Accuracy": round(comp_score, 3),
        "Overall_Score": round(overall, 3)
    }

In [23]:
# ============================================================
# 4️⃣ Dataset de perguntas e expectativas (mock)
# ============================================================

evaluation_dataset = [
    {
        "question": "Há quantos pedidos com status devolução solicitada?",
        "reference_terms": ["devolução", "pedidos"],
        "expected_values": [0], 
    },
    {
        "question": "Quais 3 vendedores tiveram o maior número de entregas atrasadas em 2018?",
        "reference_terms": ["vendedores", "entregas", "atrasadas", "2018"],
        "expected_values": [107, 94, 82], 
    },
    {
        "question": "O que acontece se o cliente pedir devolução 31 dias depois de receber o produto danificado?",
        "reference_terms": ["devolução", "produto danificado", "política de devolução", "Item não durável", "Item durável"],
        "expected_values": [],
    },
    {
        "question": "O seller 3442f8959a84dea7ee197c632cb2df15 é confiável?",
        "reference_terms": ["Atrasos", "Nota média"],
        "expected_values": [3.5],
    },
]

In [24]:
results = []

for item in evaluation_dataset:
    print(f"\nAvaliando: {item['question']}")
    try:
        response = query_chat(item["question"])
        print(f"Resposta: {response[:120]}...")
    except Exception as e:
        print(f"Erro ao consultar chat: {e}")
        response = "Erro na requisição."

    metrics = evaluate_response(response, item["reference_terms"], item["expected_values"])
    clarity_score = llm_judge(item["question"], response)

    result = {
        "Pergunta": item["question"],
        "Resposta": response,
        "Data_Consistency": metrics["Data_Consistency"],
        "Computation_Accuracy": metrics["Computation_Accuracy"],
        "Clarity_Relevance": round(clarity_score, 3),
        "Overall_Score": round((metrics["Overall_Score"] + clarity_score) / 2, 3)
    }
    results.append(result)


Avaliando: Há quantos pedidos com status devolução solicitada?


Resposta: Vou consultar o desempenho do seller solicitado e o total de pedidos com status “devolução solicitada”. Seguem as consul...

Avaliando: Quais 3 vendedores tiveram o maior número de entregas atrasadas em 2018?
Resposta: Os 3 vendedores com mais entregas atrasadas em 2018:
1. 4a3ca9315b744ce9f8e9374361493884 (Ibitinga, SP): 107 entregas at...

Avaliando: O que acontece se o cliente pedir devolução 31 dias depois de receber o produto danificado?
Resposta: Depende do tipo de item:
- Item não durável: fora do prazo de 30 dias → não elegível.
- Item durável: dentro do prazo de...

Avaliando: O seller 3442f8959a84dea7ee197c632cb2df15 é confiável?
Resposta: Não. Motivos:
- 33,33% de pedidos atrasados (limite interno: 5%)
- Nota média 3,0/5,0 (abaixo de 3,5)
- Amostra: 3 pedid...


In [25]:
df_results = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)
display(df_results)

,Pergunta,Resposta,Data_Consistency,Computation_Accuracy,Clarity_Relevance,Overall_Score
0,Há quantos pedidos com status devolução solicitada?,"Vou consultar o desempenho do seller solicitado e o total de pedidos com status “devolução solicitada”. Seguem as consultas.\nPara avaliar a confiabilidade do seller, vou calcular:\n- Total de pedidos atendidos\n- % de entregas no prazo (data_entrega <= data_estimada)\n- Nota média de review dos pedidos desse seller\nExecutando as consultas no banco.\nConsulta 1: pedidos do seller e pontualidade de entrega.\nSELECT\r\n COUNT(DISTINCT oi.order_id) AS pedidos_total,\r\n SUM(CASE WHEN o.order_delivered_customer_date IS NOT NULL AND o.order_estimated_delivery_date IS NOT NULL AND o.order_delivered_customer_date <= o.order_estimated_delivery_date THEN 1 ELSE 0 END) AS entregas_no_prazo,\r\n SUM(CASE WHEN o.order_delivered_customer_date IS NOT NULL AND o.order_estimated_delivery_date IS NOT NULL AND o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END) AS entregas_atrasadas\r\nFROM order_items oi\r\nJOIN orders o ON o.order_id = oi.order_id\r\nWHERE oi.seller_id = '3442f8959a84dea7ee197c632cb2df15';\nConsulta 2: nota média de review dos pedidos do seller.\nSELECT\r\n AVG(orv.review_score) AS nota_media,\r\n COUNT(DISTINCT orv.order_id) AS reviews_qtd\r\nFROM order_items oi\r\nJOIN order_reviews orv ON orv.order_id = oi.order_id\r\nWHERE oi.seller_id = '3442f8959a84dea7ee197c632cb2df15';\nConsulta 3: total de pedidos com status “devolução solicitada”.\nSELECT COUNT(*) AS total\r\nFROM orders\r\nWHERE order_status = 'devolucao solicitada';\nRetornando os resultados para análise.\nApresentarei os resultados consolidados sem detalhes técnicos.\n- Confiabilidade do seller 3442f8959a84dea7ee197c632cb2df15: faltam métricas no banco para concluir (ex.: volume de pedidos desse seller, taxa de atraso por pedido do seller, nota média de reviews dos pedidos do seller). Se quiser, calculo agora — confirme se avalio por: % entregas no prazo (pedido entregue <= estimado) e nota média de review por pedido com itens do seller.\n\n- Pedidos com status “devolução solicitada”: 0.",1.0,1.0,0.5,0.750
1,Quais 3 vendedores tiveram o maior número de entregas atrasadas em 2018?,"Os 3 vendedores com mais entregas atrasadas em 2018:\n1. 4a3ca9315b744ce9f8e9374361493884 (Ibitinga, SP): 107 entregas atrasadas\n2. 4869f7a5dfa277a7dca6462dcf3b52b2 (Guariba, SP): 94 entregas atrasadas\n3. 1f50f920176fa81dab994f9023523100 (São José do Rio Preto, SP): 82 entregas atrasadas",1.0,1.0,1.0,1.000
2,O que acontece se o cliente pedir devolução 31 dias depois de receber o produto danificado?,"Depende do tipo de item:\n- Item não durável: fora do prazo de 30 dias → não elegível.\n- Item durável: dentro do prazo de 90 dias → elegível, desde que apresente evidências do defeito (fotos/vídeo/descrição) e não seja mau uso.\n\nFalta: confirmar se o item é durável ou não. Próximo passo: classificar o produto e, se durável, coletar evidências.",0.4,0.0,1.0,0.600
3,O seller 3442f8959a84dea7ee197c632cb2df15 é confiável?,"Não. Motivos:\n- 33,33% de pedidos atrasados (limite interno: 5%)\n- Nota média 3,0/5,0 (abaixo de 3,5)\n- Amostra: 3 pedidos (1 atrasado)",0.5,1.0,1.0,0.875


In [26]:
print("\nMédias gerais das métricas:")
print(df_results[["Data_Consistency", "Computation_Accuracy", "Clarity_Relevance", "Overall_Score"]].mean().round(3))


Médias gerais das métricas:
Data_Consistency        0.725
Computation_Accuracy    0.750
Clarity_Relevance       0.875
Overall_Score           0.806
dtype: float64
